In [1]:
import numpy as np
import random
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import DiagonalGate
from qiskit.primitives import StatevectorEstimator 
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator

Swap Test fidelity function stolen shamelessly from Teodor

calculate_all_distances_quantumly :
Ideal Quantum Algorithm: You would load all $N$ training points into a superposition state and run a single operation to process them all simultaneously (Quantum Parallelism)
Our Implementation: Because we cannot easily load classical data into superpositions (the QRAM problem), we are forced to use a Python loop
Consequence: We are using a Quantum Computer to calculate the distance, but we are using a Classical Computer to iterate through the list

In [2]:
def classical_encoder(qc, reg, data):
    for i in range(len(data)):
        qc.ry(data[i], reg[i])

def compute_fidelity(train_point, test_point):
    no_dimensions = len(train_point)
    
    ancilla = QuantumRegister(1, name="ancilla")
    reg_a = QuantumRegister(no_dimensions, name="train")
    reg_b = QuantumRegister(no_dimensions, name="test")
    qc = QuantumCircuit(ancilla, reg_a, reg_b)

    classical_encoder(qc, reg_a, train_point)
    classical_encoder(qc, reg_b, test_point)
    qc.barrier()

    # 2. SWAP Test
    qc.h(ancilla)
    for i in range(no_dimensions):
        qc.cswap(ancilla[0], reg_a[i], reg_b[i])

    qc.h(ancilla)
    observable = SparsePauliOp("I" * (2 * no_dimensions) + "Z") 
    estimator = StatevectorEstimator()
    job = estimator.run([(qc, observable)])
    result = job.result()[0].data.evs
    
    return float(result)

def calculate_all_distances_quantumly(train_data, test_point):
    """Loops through all data and calculates 'Distance' = 1 - Fidelity"""
    distances = []
    print("Phase 1: Calculating Distances via Quantum Swap Test...")
    for i, point in enumerate(train_data):
        fid = compute_fidelity(point, test_point)
        # Convert Fidelity (Similarity) to Distance (Dissimilarity)
        dist = 1.0 - fid 
        distances.append(dist)
        print(f"   -> Point {i}: Fidelity={fid:.4f} => Distance={dist:.4f}")
    print("-" * 30)
    return distances

In [4]:
# The Mocked Oracle using the pre-calculated quantum distances
def build_oracle(distances, threshold, num_qubits):
    N = 2**num_qubits
    diagonal_elements = []
    
    for i in range(N):
        if i < len(distances) and distances[i] < threshold:
            diagonal_elements.append(-1)
        else:
            diagonal_elements.append(1)
    return DiagonalGate(diagonal_elements)

def grover_diffusion(num_qubits):
    qr = QuantumRegister(num_qubits)
    qc = QuantumCircuit(qr)
    qc.h(qr)
    qc.x(qr)
    qc.h(qr[-1])
    qc.mcx(qr[:-1], qr[-1])
    qc.h(qr[-1])
    qc.x(qr)
    qc.h(qr)
    return qc.to_gate(label="Diffusion")

def calculate_grover_iterations(n):
    grover_iterations = int((np.pi / 4) * np.sqrt(n))
    grover_iterations = max(1, grover_iterations)
    return grover_iterations 

def calculate_num_qubits(n):
    return int(np.ceil(np.log2(n)))

# dur hoyer returns the index of the minimum value in the distances array
# shots is the number of times the circuit is run to get the final probability distribution  (it is passed to the simulator)
def run_durr_hoyer(distances, shots=1024, grover_iterations=None, dur_hoyer_iterations=15, num_qubits=None):
    n = len(distances)
    num_qubits = num_qubits or calculate_num_qubits(n)
    grover_iterations = grover_iterations or calculate_grover_iterations(n)
    sim = AerSimulator()
    
    valid_distances = [d for d in distances if d != float('inf')]
    if not valid_distances:
        return -1
        
    current_threshold = random.choice(valid_distances)
    best_index = distances.index(current_threshold) 
    
    # Run the search steps
    # The Dürr-Høyer algorithm works by picking a random index, creating a threshold, and then using Grover's search to find any index with a value smaller than that threshold. When it finds one, it lowers the threshold and repeats.
    # This loop tries to "beat the current best" up to 15 times.
    for step in range(dur_hoyer_iterations): 
        qr = QuantumRegister(num_qubits)
        cr = ClassicalRegister(num_qubits)
        qc = QuantumCircuit(qr, cr)
        qc.h(qr)
        
        oracle = build_oracle(distances, current_threshold, num_qubits)

        # The standard Grover algorithm requires repeating the [Oracle + Diffusion] sequence multiple times to amplify the probability of the correct answer. If you only do it once, the probability might only rise to 20% or 30%, meaning you are measuring "noise" most of the time.
        for _ in range(grover_iterations):
            qc.append(oracle, qr)
            qc.append(grover_diffusion(num_qubits), qr)

        qc.measure(qr, cr)
        
        result = sim.run(transpile(qc, sim), shots=shots).result()
        counts = result.get_counts()
        if not counts: continue
            
        most_likely = max(counts, key=counts.get)
        idx = int(most_likely, 2)
        
        if idx < len(distances):
            val = distances[idx]
            if val < current_threshold:
                current_threshold = val
                best_index = idx
    
    return best_index

# uses the Dur Hoyer to implement a k-minima, does the masking too so the best index is not found again
def quantum_knn(distances, k):
    working_distances = list(distances).copy()
    found_indices = []
    found_values = []
    
    print(f"\nPhase 2: Running Dürr-Høyer Search on Quantum Distances...")
    print("-" * 30)
    
    for i in range(k):
        print(f"Searching for neighbor #{i+1}...")
        idx = run_durr_hoyer(working_distances)
        
        if idx == -1 or working_distances[idx] == float('inf'):
            print("   -> No better neighbor found.")
            break
            
        val = working_distances[idx]
        found_indices.append(idx)
        found_values.append(val)
        print(f"   -> Found Index {idx} (Distance: {val:.4f})")
        
        working_distances[idx] = float('inf')
        
    return found_indices, found_values    

In [5]:
train_data = [
        [0.1],          # Close to 0
        [np.pi],        # Far (180 degrees opposite)
        [0.2],          # Very close to 0
        [np.pi / 2]     # Orthogonal (90 degrees)
    ]
    
# we want to find neighbors for a point near 0.15
test_point = [0.15]
k = 2

quantum_distances = calculate_all_distances_quantumly(train_data, test_point)

# find the Minimums using Dürr-Høyer
inds, vals = quantum_knn(quantum_distances, k)

print(f"\nFinal Result Indices: {inds}")
print(f"These indices correspond to training data: {[train_data[i] for i in inds]}")

Phase 1: Calculating Distances via Quantum Swap Test...
   -> Point 0: Fidelity=0.9994 => Distance=0.0006
   -> Point 1: Fidelity=0.0056 => Distance=0.9944
   -> Point 2: Fidelity=0.9994 => Distance=0.0006
   -> Point 3: Fidelity=0.5747 => Distance=0.4253
------------------------------

Phase 2: Running Dürr-Høyer Search on Quantum Distances...
------------------------------
Searching for neighbor #1...
   -> Found Index 0 (Distance: 0.0006)
Searching for neighbor #2...
   -> Found Index 2 (Distance: 0.0006)

Final Result Indices: [0, 2]
These indices correspond to training data: [[0.1], [0.2]]


In [15]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Load Iris Data
iris = load_iris()
X = iris.data
y = iris.target
target_names = iris.target_names

# 2. Scale to [0, Pi] for Quantum Rotation Encoding (Ry gates)
# CRITICAL: Raw values like 5.1 or 7.9 would loop around the Bloch sphere 
# multiple times. We must map them to 0 to 180 degrees (0 to pi).
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X)

# 3. Split Data
# We keep the training set small (15 samples) so the simulation runs quickly.
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, train_size=15, test_size=3, random_state=42, stratify=y
)

# 4. Format Data for the Quantum Functions
# The QKNN functions expect a list of numpy arrays
train_data = [np.array(pt) for pt in X_train]
test_data = [np.array(pt) for pt in X_test]

# Convert numeric labels (0, 1, 2) to string names ('setosa', etc.)
train_labels = [target_names[i] for i in y_train]
test_labels = [target_names[i] for i in y_test]

print(f"Data Prepared Successfully:")
print(f" - Training Set: {len(train_data)} samples")
print(f" - Test Set:     {len(test_data)} samples")
print(f" - Features:     {len(train_data[0])} (Requires 4 qubits for encoding)")

Data Prepared Successfully:
 - Training Set: 15 samples
 - Test Set:     3 samples
 - Features:     4 (Requires 4 qubits for encoding)


In [17]:
from collections import Counter

print(f"Dataset prepared. Running predictions on {len(test_data)} test points...\n")

for i, test_point in enumerate(test_data):
    print(f"--- Test Point {i+1} (Actual: {test_labels[i]}) ---")
    
    # 1. Calculate Distances (Quantumly - Swap Test)
    q_dists = calculate_all_distances_quantumly(train_data, test_point)
    
    # 2. Find Neighbors (Quantum Search - Dürr-Høyer)
    inds, vals = quantum_knn(q_dists, k=3)
    
    # 3. Check result
    neighbor_labels = [train_labels[idx] for idx in inds]
    print(f"    -> Neighbors Found: {neighbor_labels}")
    
    # --- ADDED: Majority Voting ---
    # Finds the most common label among the 3 neighbors
    prediction = Counter(neighbor_labels).most_common(1)[0][0]
    
    if prediction == test_labels[i]:
        print(f"    -> Final Prediction: {prediction} ✅")
    else:
        print(f"    -> Final Prediction: {prediction} ❌")
    print("-" * 30)

Dataset prepared. Running predictions on 3 test points...

--- Test Point 1 (Actual: setosa) ---
Phase 1: Calculating Distances via Quantum Swap Test...
   -> Point 0: Fidelity=0.0830 => Distance=0.9170
   -> Point 1: Fidelity=0.2781 => Distance=0.7219
   -> Point 2: Fidelity=0.6505 => Distance=0.3495
   -> Point 3: Fidelity=0.2329 => Distance=0.7671
   -> Point 4: Fidelity=0.9706 => Distance=0.0294
   -> Point 5: Fidelity=0.0032 => Distance=0.9968
   -> Point 6: Fidelity=0.9546 => Distance=0.0454
   -> Point 7: Fidelity=0.0620 => Distance=0.9380
   -> Point 8: Fidelity=0.3052 => Distance=0.6948
   -> Point 9: Fidelity=0.8130 => Distance=0.1870
   -> Point 10: Fidelity=0.2649 => Distance=0.7351
   -> Point 11: Fidelity=0.7629 => Distance=0.2371
   -> Point 12: Fidelity=0.0011 => Distance=0.9989
   -> Point 13: Fidelity=0.0388 => Distance=0.9612
   -> Point 14: Fidelity=0.0232 => Distance=0.9768
------------------------------

Phase 2: Running Dürr-Høyer Search on Quantum Distances...
-